In [1]:
!python --version

Python 3.11.13


공개 경로 사용

In [2]:
!pip uninstall -y keras keras-nightly keras3 tf_keras tf-nightly tf-keras


Found existing installation: keras 2.15.0
Uninstalling keras-2.15.0:
  Successfully uninstalled keras-2.15.0


In [3]:
import os, sys
os.environ.pop("TF_USE_LEGACY_KERAS", None)

# 혹시 이전에 주입했던 내부 모듈 키가 남아있다면 삭제
for k in [
    "tensorflow.python.keras.layers",
    "tensorflow.python.keras.initializers",
    "tensorflow.python.ops.init_ops_v2",
]:
    if k in sys.modules:
        del sys.modules[k]


In [4]:
!pip install -q "tensorflow==2.15.0.post1"

In [ ]:
!pip install shap==0.45.1

재시작

In [1]:
import tensorflow as tf, importlib.util, os
print("TF:", tf.__version__)
print("TF_USE_LEGACY_KERAS =", os.environ.get("TF_USE_LEGACY_KERAS"))
print("has tensorflow.keras ?",
      importlib.util.find_spec("tensorflow.keras") is not None)


TF: 2.15.0
TF_USE_LEGACY_KERAS = None
has tensorflow.keras ? True


In [2]:
!pip install -q --no-deps "deepctr==0.9.3"

In [3]:
import importlib, re, shutil
from pathlib import Path

# 1) deepctr 설치/위치 확인
m = importlib.import_module("deepctr")
pkg_dir = Path(m.__file__).parent  # e.g. /usr/local/lib/python3.11/dist-packages/deepctr
print("DeepCTR package dir:", pkg_dir)

# 2) 치환 규칙 (프라이빗 → 공개)
#    - 최대한 보수적으로, 대표적으로 문제되는 임포트만 교체
patterns = [
    # layers
    (r"from\s+tensorflow\.python\.keras\.layers\s+import\s+",  r"from tensorflow.keras.layers import "),
    (r"import\s+tensorflow\.python\.keras\.layers\s+as\s+",    r"import tensorflow.keras.layers as "),
    # initializers
    (r"from\s+tensorflow\.python\.keras\.initializers\s+import\s+", r"from tensorflow.keras.initializers import "),
    (r"import\s+tensorflow\.python\.keras\.initializers\s+as\s+",   r"import tensorflow.keras.initializers as "),
    # init_ops_v2 → keras.initializers
    (r"from\s+tensorflow\.python\.ops\.init_ops_v2\s+import\s+",    r"from tensorflow.keras.initializers import "),
    # 남아있는 tensorflow.python.keras.* 전반 치환(최후방어)
    (r"tensorflow\.python\.keras\.", r"tensorflow.keras."),
]

changed_files = []
for p in pkg_dir.rglob("*.py"):
    txt = p.read_text(encoding="utf-8")
    orig = txt
    for pat, rep in patterns:
        txt = re.sub(pat, rep, txt)
    if txt != orig:
        # 백업 1회 저장
        bak = p.with_suffix(p.suffix + ".bak")
        if not bak.exists():
            shutil.copy2(p, bak)
        p.write_text(txt, encoding="utf-8")
        changed_files.append(str(p.relative_to(pkg_dir)))

print(f"Patched {len(changed_files)} file(s):")
for f in changed_files:
    print(" -", f if len(f) < 120 else f[:117] + "...")

print("✅ Patch complete. (라이브러리 파일 수정 완료)")


DeepCTR package dir: /usr/local/lib/python3.11/dist-packages/deepctr
Patched 0 file(s):
✅ Patch complete. (라이브러리 파일 수정 완료)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from deepctr.feature_column import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr.models import DeepFM
import numpy as np


# 데이터 준비



In [5]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

train = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/train_part1.parquet' , engine = 'pyarrow')
test  = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/test.parquet'        , engine = 'pyarrow')
valid = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/train_part3.parquet' , engine = 'pyarrow')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


test 열에 없는 열 train에서 버리기

In [6]:
target = 'clicked'

cols_to_keep = [c for c in train.columns if c in test.columns or c == target]

train = train[cols_to_keep]
valid = valid[cols_to_keep]

train , test 데이터 타입 맞추기

- test 데이터 id열 제외하고 float32로




In [7]:
cols_to_convert = [c for c in test.columns if c not in ['seq' ,'ID']]
test[cols_to_convert]  = test[cols_to_convert].astype("float32")

플래그 설정

In [8]:
train["is_train"] = 1
test["is_train"]  = 0
valid["is_train"] = 2

test_id = test["ID"].copy()

/tmp/ipython-input-8-816344154.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["is_train"]  = 0


In [9]:
all_data = pd.concat([train, test,valid], ignore_index=True)

In [10]:
df = all_data.copy()

# 개선점

### 검증셋 누수 막기 대작전

>

    검증셋 누수를 막는다는 건
    1. 그룹 단위 분리 (같은 user/ad/time이 겹치지 않게)

    2. 시간 순서 보존
    
    3. 전처리 독립성 보장

>

    범주형 : target 기반으로 파생된 값
      ? l_feat_14    (Ads set .  어떻게 생성됐는지 확신X. 이것도 임베딩으로 관리해서 누수까진 아님)
      ? inventory_id (누수는 아님. 다만 같은 ID가 train/valid에 동시에 있으면 암기 효과로 검증이 과대평가 -> StratifiedGroupKFold)

    연속형 : target 기반 통계
      history_a_*  (타깃 기반 통계일 가능성 큼, 피처간 상관 0.98~0.99 ⇒ OOF 재계산 필수 후보. 대표 1-2개만 사용)
      다른 연속형은 일단 그대로 (누수 판단할 근거 없음. 이후 importance로 검증)

inventory_id = 92 , 21 이상치로 간주후 삭제하려 했으나 아래 같은 방식으로 하면 test 데이터도 함께 날아가는 이슈 발생

따라서 어차피 unk(0)로 학습해야 할 바엔 표본이 충분한 92는 그대로 학습 , 21은 unk(0)으로 일단 ㄱㄱ

In [11]:
df = df.drop(['l_feat_20' , 'l_feat_23','l_feat_2','l_feat_24'], axis = 1 )
# df = df.drop(['feat_a_15' , 'feat_a_17'])
# df = df.drop(['history_a_1' , 'history_a_2' , 'history_a_3'], axis= 1)
# df = df[~df['inventory_id'].isin([92, 21])]

In [12]:
df.info(verbose = True , show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7935066 entries, 0 to 7935065
Data columns (total 117 columns):
 #    Column        Non-Null Count    Dtype  
---   ------        --------------    -----  
 0    gender        7934130 non-null  float32
 1    age_group     7934130 non-null  float32
 2    inventory_id  7935066 non-null  float32
 3    day_of_week   7935066 non-null  float32
 4    hour          7935066 non-null  float32
 5    seq           7935066 non-null  object 
 6    l_feat_1      7935066 non-null  float32
 7    l_feat_3      7935066 non-null  float32
 8    l_feat_4      7935066 non-null  float32
 9    l_feat_5      7935066 non-null  float32
 10   l_feat_6      7935066 non-null  float32
 11   l_feat_7      7935066 non-null  float32
 12   l_feat_8      7934130 non-null  float32
 13   l_feat_9      7935066 non-null  float32
 14   l_feat_10     7935066 non-null  float32
 15   l_feat_11     7935066 non-null  float32
 16   l_feat_12     7935066 non-null  float32
 17   l_feat

In [13]:
is_train_num = pd.to_numeric(df["is_train"], errors="coerce")
te_mask = is_train_num.eq(0)
tr_mask = is_train_num.eq(1)
va_mask = is_train_num.eq(2)

In [14]:
# 전역 CTR + 베타-스무딩
p_global = df.loc[tr_mask, "clicked"].mean()
prior_s  = 300
alpha, beta = p_global*prior_s, (1-p_global)*prior_s

grp = df.loc[tr_mask, ["inventory_id","clicked"]].dropna(subset=["inventory_id"]).groupby("inventory_id")
cnt = grp.size()
pos = grp["clicked"].sum()
post_ctr = (pos.add(alpha, fill_value=0)) / (cnt.add(prior_s, fill_value=0))

# # 보존 기준(튜닝 가능)-----------------> train 50%만 살아남음
# min_n = 500      # 표본 수 기준
# delta = 0.010    # 전역 CTR 대비 1.0%p 이상 차이면 보존
# keep_ids = post_ctr[(cnt >= min_n) & ((post_ctr - p_global).abs() >= delta)].index

# string 으로 단단하게 고정
tr_ids = df.loc[tr_mask, "inventory_id"].astype("string")
vc = tr_ids.value_counts()

target_cov = 0.95  # ---------------------> 임시 해결 방법 :95% 커버 목표 (튜닝)
cum = vc.cumsum() / vc.sum()
K = max(1, int((cum <= target_cov).sum()))

keep_ids = set(vc.index[:K])
mp_inv = {k:i+1 for i,k in enumerate(sorted(keep_ids))}
inv_vocab_size = len(mp_inv) + 1  # 0=UNK

# train-only map: 보존 ID만 개별 임베딩, 나머지는 UNK=0
# mp_inv = {v:i+1 for i, v in enumerate(sorted(keep_ids))}
# inv_vocab_size = (max(mp_inv.values()) + 1) if mp_inv else 1  # UNK만 있어도 최소 1

# 보조 플래그(선택) — 모델에 Dense로 같이 넣으면 도움될 수 있음
df["inv_is_rare"]  = (~df["inventory_id"].isin(keep_ids)).astype("int8")
df["inv_is_92_21"] = df["inventory_id"].isin([92,21]).astype("int8")


## seq_padded,seq_len,vocab_size

단순 임베딩+패딩만 하면 효과 미미함 → attention, pooling 전략 필요.

진단:
>

    DIN/BST 모델에서 seq_length, mask, embedding dimension 설정 확인

    padding vs truncation 정책 적절성 확인

    sequence 내 token 분포와 CTR lift 간 상관분석 수행

    max_len : 800
    topk : 1000
    min_count = 500

In [15]:
MAX_LEN = 150
PAD_ID = 0  # 0은 PAD, 실제 토큰은 +1 오프셋

def build_seq_padded_len(series: pd.Series, max_len: int, *, offset: int = 1, ignore_neg: bool = True):
    """
    series: 콤마 구분 문자열("9,18,269,...") 컬럼
    offset=1: 모델 전처리처럼 +1 오프셋(0은 PAD로 예약)
    return: seq_padded(int32, [N,max_len]), seq_len(int32, [N]), vocab_size(int)
    """
    # 미리 결과 배열을 한 번에 할당(메모리/속도 핵심)
    N = len(series)
    seq_padded = np.zeros((N, max_len), dtype=np.int32)
    seq_len    = np.zeros(N, dtype=np.int32)
    max_id     = 0

    # 판다스 오버헤드 줄이기: 바로 넘파이 배열로
    # astype(str)을 쓰면 NaN -> 'nan' 문자열이 되므로, 아래에서 비어 있으면 건너뜀
    vals = series.to_numpy(copy=False)

    for i in range(N):
        s = vals[i]
        if s is None or (isinstance(s, float) and np.isnan(s)):
            # 빈 시퀀스
            continue

        # 문자열로 캐스팅 (np.fromstring은 공백을 무시하므로 replace 불필요)
        text = s if isinstance(s, str) else str(s)
        if not text:
            continue

        # C 가속 파싱: 매우 빠름. 실패하면 size=0
        arr = np.fromstring(text, sep=',', dtype=np.int64)
        if arr.size == 0:
            continue

        if ignore_neg:
            # 음수 제거(있다면)
            arr = arr[arr >= 0]
            if arr.size == 0:
                continue

        if offset:
            # +1 오프셋 (0=PAD 유지)
            arr = arr + offset

        # 트렁케이팅: 최신 항목을 남기고 앞을 자름 (pre-truncating)
        L = arr.size
        if L > max_len:
            arr = arr[-max_len:]
            L = max_len

        # 패딩된 행에 앞쪽부터 복사 (post-padding)
        # arr는 int64이므로 복사 시 자동 캐스팅 → 비용 적음
        seq_padded[i, :L] = arr
        seq_len[i] = L

        # vocab_size 계산용 최대 id 갱신 (한 번에 끝)
        amax = int(arr.max()) if L > 0 else 0
        if amax > max_id:
            max_id = amax

    vocab_size = int(max_id + 1)  # PAD 포함
    return seq_padded, seq_len, vocab_size


seq_padded, seq_len, vocab_size = build_seq_padded_len(df["seq"], MAX_LEN, offset=1, ignore_neg=True)


In [16]:
PAD_ID   = 0
MAX_LEN  = 150
OFFSET   = 1


tr_mask = df["is_train"].eq(1)
te_mask = df["is_train"].eq(0)
va_mask = df["is_train"].eq(2)


# 1) train만으로 vocab “fit”
seq_tr_pad, seq_tr_len, vocab_seq = build_seq_padded_len(
    df.loc[tr_mask, "seq"], MAX_LEN, offset=OFFSET, ignore_neg=True
)

# 2) test는 같은 규칙으로 transform만 + OOV 클램핑
seq_te_pad, seq_te_len, _ = build_seq_padded_len(
    df.loc[te_mask, "seq"], MAX_LEN, offset=OFFSET, ignore_neg = True
)
seq_va_pad, seq_va_len, _= build_seq_padded_len(
    df.loc[va_mask, "seq"], MAX_LEN , offset=OFFSET, ignore_neg= True
)


# train에서 결정한 vocab_seq를 기준으로, 범위 밖 토큰(>= vocab_seq)은 0으로
seq_tr_pad = np.where(seq_tr_pad < vocab_seq, seq_tr_pad, PAD_ID).astype("int32")
seq_te_pad = np.where(seq_te_pad < vocab_seq, seq_te_pad, PAD_ID).astype("int32")
seq_va_pad = np.where(seq_va_pad < vocab_seq, seq_va_pad, PAD_ID).astype("int32")


# 길이 저장(원하면 df에도 반영)
df.loc[tr_mask, "seq_len"] = seq_tr_len
df.loc[te_mask, "seq_len"] = seq_te_len
df.loc[va_mask, "seq_len"] = seq_va_len


## 피처 열 생성


> DeepCTR에서 SparseFeat는 내부적으로 Embedding Lookup을 하기 때문에, 입력값은 반드시 0 ~ (vocabulary_size-1) 범위의 연속 정수 인덱스여야 한다.

>
    SparseFeat("inventory_id", vocabulary_size=16) 같이 정의하면, DeepCTR은 입력값을 0~15 정수 인덱스로 간주한다.

    그런데 실제 데이터는 2.0, 36.0, 37.0, …, 95.0처럼 흩어져 있음.

    이걸 그대로 Embedding lookup에 넣으면 → 인덱스 범위 초과 오류 또는 embedding index mismatch 발생.

> label encoding으로 맞춘다.


<br>


순서형의 경우 임베딩은 순서 정보를 기억하지 않는다.
>
    조회만 한다: e_k = Embedding[k] — k는 의미 있는 수가 아니라 “키”.

    순열 불변성: 첫 레이어가 ŷ = W·e_k + b일 때, 라벨을 임의로 섞고(순열 P) 임베딩과 가중치를 같이 섞으면 ŷ가 그대로.
    즉 모델은 라벨 순서에 무관하게 동치 해를 가짐.

    그래디언트 독립: 각 카테고리 벡터가 독립적으로 업데이트되어 연속성/단조성이 보장되지 않음. “2는 1과 3 사이”라는 규칙을 스스로 학습하리란 보장이 없다.

    그래서 임베딩으로 처리하면 ‘명목형처럼’ 취급되고, 순서(ordinal) 정보는 구조적으로 전달되지 않는다.

> 두 개의 표현을 동시에 사용 - SparseFeat(범주형) 과 DenseFeat(순서형 숫자) 둘 다 넣기.



<BR>

hour , day_of_week 열에 관하여

>

    day_of_week: 순서 중요, 월~일 주기성 있음

    hour: 시간대도 순서형이라 DenseFeat 가능. 다만 주기성(23시→0시)이 있어서 사인/코사인 변환.




< 계획 >

1. 결측치 보강


2. **hour, day_of_week**

 >
    사인/코사인 변환 + 이중표현(Sparse: 원래 카테고리도 함께 넣어 요일/시간대별 개별 패턴 포착)

    Sparse: 원래 카테고리도 함께 넣어 요일/시간대별 개별 패턴 포착.
>
    장점: 주기성(23→0의 연속성) + 카테고리별 임베딩 패턴을 동시에 잡는다.



3. 연속형 추정 열들

>

    l_feat_3, l_feat_27, feat_e_4, feat_a_1, feat_a_3, feat_a_4, feat_a_8, feat_a_13, feat_a_16, feat_a_18
>
    Dense: 결측치 imputer(median 등) → MinMaxScaler 후 그대로 입력.

    Sparse(해시): 결측치 보강 후 -> 바로  처리


> 이중화의 경우 과적합의 위험이 있으니 embedding_dim을 작게(4~8), dnn_dropout, l2_reg_embedding 등을 적절히 사용

In [17]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

####### 1번
ord_cols  = ["hour", "day_of_week", "age_group"] # age_group : cont 그룹으로 넣으면 2.5 같은 그룹 생성됨

# 순서형
cont_cols = [
              'l_feat_3','l_feat_27','feat_e_4',
              'feat_a_1','feat_a_3','feat_a_4',
              'feat_a_6','feat_a_7','feat_a_8',
              'feat_a_9','feat_a_10','feat_a_11',
              'feat_a_13','feat_a_16','feat_a_18'
              ]




cat_cols =  ["gender", "inventory_id", "l_feat_14","l_feat_14_hash"]

seq_col = 'seq'
label_col = 'clicked'
seq_len_col = 'seq_len'

ban = set(ord_cols + cont_cols + cat_cols + [seq_col, label_col, seq_len_col, "is_train", "ID"])

# 진또베기 연속형들
real_cont_cols = [c for c in df if c not in ban]

# 순서형(순서 학습을 위한 dense) +  찐연속형
cont_all_cols = cont_cols + real_cont_cols

# dense용 전체 (ord + cont + 찐연속형)
num_all_cols = ord_cols + cont_all_cols


# train / test 분리
tr_mask = df["is_train"].eq(1)
te_mask = df["is_train"].eq(0)
va_mask = df["is_train"].eq(2)


# train으로만 fit : test 결측치 대체할 통계 학습
imp_ord  = SimpleImputer(strategy="most_frequent").fit(df.loc[tr_mask , ord_cols])
imp_cnt  = SimpleImputer(strategy="most_frequent").fit(df.loc[tr_mask ,cont_cols])
imp_real = SimpleImputer(strategy="median").fit(df.loc[tr_mask, real_cont_cols])

# 기준값: impute된 값을 기준으로 test/valid 결측치를 채움
Xord_tr  = imp_ord.transform(df.loc[tr_mask,ord_cols])
Xcnt_tr  = imp_cnt.transform(df.loc[tr_mask,cont_cols])
Xreal_tr = imp_real.transform(df.loc[tr_mask,real_cont_cols])


Xnum_tr = np.hstack([Xord_tr,Xcnt_tr,Xreal_tr])


# Dense용 MinMax 스케일러 - train 단계에서 fit만해서 따로 저장
scaler = MinMaxScaler().fit(Xnum_tr)


# age 임베딩
AGE_DOMAIN = [1,2,3,4,5,6,7,8]
mp_age = {v: v for v in AGE_DOMAIN}   # 아이덴티티 매핑 (0은 UNK로 예약)
age_vocab_size = max(AGE_DOMAIN) + 1






# train에서만 mp 생성 - cont_cols 컬럼을 sparse에 넣기 위함 ( cont_cols만 sparse에도 쓰도록 고정 )

CNT_FIT_COLS = cont_cols
_idx = {c: i for i, c in enumerate(CNT_FIT_COLS)}


def _round_vals(x):  # 정수 bin이면 0자리 반올림 -----------------------> 의문인 점 : 반올림하면 dense와 sparse 둘 다 넣는 의미가 있나? 동일하게 맞추는 건 이해하지만 수치 그 자체로 의미있다면?
    return np.round(x, 0).astype("int64")

# map_cols를 cont_cols로 고정 : 이 열들만 ~.
map_cols = CNT_FIT_COLS

# 컬럼별 정적 매핑(1..K), 0은 UNK 슬롯
mp_cont = {
    c: {v: i+1 for i, v in enumerate(np.unique(_round_vals(Xcnt_tr[:, _idx[c]])))}
    for c in map_cols
}

# vocab 크기 dict도 같이 보관 - > DeepFM 정의에 사용하기 위함
vocab_cont = {c: len(mp_cont[c]) + 1 for c in map_cols}  # 0=UNK 포함


test 데이터에 결측치가 없어도 전처리 산출물에 정의를 해야함 -> 모델 입력 만들 때 keyEorror

> 결측치 유무는 상관 없이 정의하면 입력에도 존재해야 한다.


>

    age_group, inventory_id 등은 train에서만 mp = {value: i+1} 만들고,
    test는 map(...).fillna(0)로 OOV를 0에 보냄.

    vocabulary_size = max(mp.values()) + 1 로 맞춤

## l_feat_14의 cold start 문제

1. l_feat_14 : 문자열 대신 int32 해시로 미리 만들어두기

2. tail cutoff로 embedding을 정리

In [18]:
from __future__ import annotations

import json
import hashlib
from dataclasses import dataclass, asdict
from typing import List, Any

import numpy as np
import pandas as pd


def stable_hash_str(s: str) -> int:
    """Stable 64-bit unsigned hash for strings."""
    h = hashlib.blake2b(s.encode("utf-8"), digest_size=8).hexdigest()
    return int(h, 16)


def _reserve_unk_offset(h: int, buckets: int) -> int:
    """Map raw hash to [1, buckets-1] reserving 0 for UNK."""
    if buckets < 2:
        raise ValueError("buckets must be >= 2 when reserving UNK=0")
    return (h % (buckets - 1)) + 1


@dataclass
class CrossArtifacts:
    l_col: str
    inv_col: str
    head_coverage: float
    top_n_inventory: int
    min_inv_freq: int
    head_l14: List[str]
    top_inventory: List[str]
    l14_buckets: int
    cross_buckets: int
    unk_token: str = "__UNK__"
    other_bucket_token: str = "__OTHER__"

    def to_json(self) -> str:
        return json.dumps(asdict(self), ensure_ascii=False, indent=2)

    @staticmethod
    def from_json(s: str) -> "CrossArtifacts":
        return CrossArtifacts(**json.loads(s))


def _is_nan_like(x: Any) -> bool:
    return x is None or (isinstance(x, float) and np.isnan(x))


def pick_head_by_coverage(train: pd.Series, coverage: float) -> List[str]:
    """Tokens whose cumulative freq covers `coverage` (NaN excluded)."""
    s = train.dropna().astype(str)
    vc = s.value_counts()
    if vc.empty:
        return []
    cum = vc.cumsum() / vc.sum()
    head = vc.index[cum <= coverage].tolist() or [vc.index[0]]
    return [str(x) for x in head]


def pick_top_inventory(train_inv: pd.Series, top_n: int, min_freq: int) -> List[str]:
    """Top-N inventory ids by freq (>= min_freq, NaN excluded)."""
    s = train_inv.dropna().astype(str)
    vc = s.value_counts()
    if min_freq > 1:
        vc = vc[vc >= min_freq]
    return vc.head(top_n).index.astype(str).tolist()


def build_artifacts(
    train_df: pd.DataFrame,
    l_col: str = "l_feat_14",
    inv_col: str = "inventory_id",
    head_coverage: float = 0.90,
    top_n_inventory: int = 8,
    min_inv_freq: int = 1,
    l14_buckets: int = 200_000,
    cross_buckets: int = 131_072,
) -> CrossArtifacts:
    """Build train-only artifacts."""
    if l_col not in train_df.columns or inv_col not in train_df.columns:
        raise ValueError("Required columns not found in train_df")
    if l14_buckets < 2 or cross_buckets < 2:
        raise ValueError("bucket sizes must be >= 2 (UNK=0 reserved)")

    head_l14 = pick_head_by_coverage(train_df[l_col], head_coverage)
    top_inv = pick_top_inventory(train_df[inv_col], top_n_inventory, min_inv_freq)

    return CrossArtifacts(
        l_col=l_col,
        inv_col=inv_col,
        head_coverage=head_coverage,
        top_n_inventory=top_n_inventory,
        min_inv_freq=min_inv_freq,
        head_l14=head_l14,
        top_inventory=top_inv,
        l14_buckets=int(l14_buckets),
        cross_buckets=int(cross_buckets),
    )


def map_l14_token(token: Any, art: CrossArtifacts, head_set: set[str] | None = None) -> str:
    """Head token → itself, else UNK. NaN → UNK."""
    if _is_nan_like(token):
        return art.unk_token
    t = str(token)
    hs = head_set if head_set is not None else set(art.head_l14)
    return t if t in hs else art.unk_token


def hash_l14(token: Any, art: CrossArtifacts, head_set: set[str] | None = None) -> int:
    """l14_hash with UNK=0 reserved; heads map to [1..B-1]."""
    mapped = map_l14_token(token, art, head_set)
    if mapped == art.unk_token:
        return 0
    return _reserve_unk_offset(stable_hash_str(mapped), art.l14_buckets)


def hash_cross(
    l14: Any,
    inv: Any,
    art: CrossArtifacts,
    head_set: set[str] | None = None,
    topi_set: set[str] | None = None,
) -> int:
    """l14_inv_cross with UNK=0 reserved per rules."""
    hs = head_set if head_set is not None else set(art.head_l14)
    ts = topi_set if topi_set is not None else set(art.top_inventory)

    l_is_nan = _is_nan_like(l14)
    i_is_nan = _is_nan_like(inv)
    l_str = None if l_is_nan else str(l14)
    i_str = None if i_is_nan else str(inv)

    if (l_is_nan or l_str not in hs) and i_is_nan:
        return 0

    if l_str is not None and l_str in hs:
        key = f"{l_str}|{i_str}"
    else:
        key = f"{art.unk_token}|{i_str}" if (i_str is not None and i_str in ts) else f"{art.unk_token}|{art.other_bucket_token}"

    return _reserve_unk_offset(stable_hash_str(key), art.cross_buckets)


def transform_with_artifacts(
    df: pd.DataFrame,
    art: CrossArtifacts,
    out_l14_hash: str = "l14_hash",
    out_cross_hash: str = "l14_inv_cross",
) -> pd.DataFrame:
    """Add hashed columns to df using artifacts."""
    if art.l_col not in df.columns or art.inv_col not in df.columns:
        raise ValueError("Required columns not found in df")

    head_set = set(art.head_l14)
    topi_set = set(art.top_inventory)

    lvals = df[art.l_col].to_numpy(copy=False)
    ivals = df[art.inv_col].to_numpy(copy=False)

    lhash = np.fromiter((hash_l14(l, art, head_set) for l in lvals), dtype=np.int64, count=len(lvals))
    chash = np.fromiter((hash_cross(l, i, art, head_set, topi_set) for l, i in zip(lvals, ivals)), dtype=np.int64, count=len(lvals))

    out = df.copy()
    out[out_l14_hash] = lhash
    out[out_cross_hash] = chash
    return out


In [19]:
tr_mask = df["is_train"].eq(1)
te_mask = df["is_train"].eq(0)
va_mask = df["is_train"].eq(2)

# 1) 아티팩트는 오직 TRAIN 행에서만!
art = build_artifacts(
    df.loc[tr_mask],
    l_col="l_feat_14",
    inv_col="inventory_id",
    head_coverage=0.90,
    top_n_inventory=8,
    min_inv_freq=10,          # 최소 등장 횟수
    l14_buckets=200_000,
    cross_buckets= 131_072
)

# 2) 동일 규칙으로 각 split을 변환
tr_out = transform_with_artifacts(df.loc[tr_mask], art)  # returns copy with 2 new cols
va_out = transform_with_artifacts(df.loc[va_mask], art)
te_out = transform_with_artifacts(df.loc[te_mask], art)

# 3) 원본 df에 해시 컬럼만 안전하게 다시 꽂기
cols_new = ["l14_hash", "l14_inv_cross"]
df.loc[tr_mask, cols_new] = tr_out[cols_new].to_numpy()
df.loc[va_mask, cols_new] = va_out[cols_new].to_numpy()
df.loc[te_mask, cols_new] = te_out[cols_new].to_numpy()

# 4) 빠른 검증(UNK=0 예약 및 범위 확인)
assert (df.loc[tr_mask, "l14_hash"] == 0).any() or len(art.head_l14) > 0
assert (0 <= df["l14_hash"]).all() and (df["l14_hash"].max() <= art.l14_buckets - 1)
assert (0 <= df["l14_inv_cross"]).all() and (df["l14_inv_cross"].max() <= art.cross_buckets - 1)

print(
    "l14_hash nunique:",
    df["l14_hash"].nunique(),
    " | l14_inv_cross nunique:",
    df["l14_inv_cross"].nunique(),
)


l14_hash nunique: 308  | l14_inv_cross nunique: 3975


In [20]:
# 1) UNK 비율 (헤드 컷오프 타이트/느슨 판단용)
def rate_unk(s):
    return (s == 0).mean()

print("UNK rate (train):", rate_unk(df.loc[tr_mask, "l14_hash"]))
print("UNK rate (valid):", rate_unk(df.loc[va_mask, "l14_hash"]))
print("UNK rate (test) :", rate_unk(df.loc[te_mask, "l14_hash"]))

# 2) 교차도 UNK 비율
print("Cross UNK rate (train):", rate_unk(df.loc[tr_mask, "l14_inv_cross"]))
print("Cross UNK rate (valid):", rate_unk(df.loc[va_mask, "l14_inv_cross"]))
print("Cross UNK rate (test) :", rate_unk(df.loc[te_mask, "l14_inv_cross"]))

# 3) 분포 스냅샷 (상위 빈도 키들)
for col in ["l14_hash", "l14_inv_cross"]:
    print(col, df.loc[tr_mask, col].value_counts().head(10))


UNK rate (train): 0.10003448937773984
UNK rate (valid): 0.10079706406257656
UNK rate (test) : 0.31250548354021285
Cross UNK rate (train): 0.0
Cross UNK rate (valid): 0.0
Cross UNK rate (test) : 0.0
l14_hash l14_hash
0.0         320499
147575.0     93420
21850.0      92113
160002.0     90348
24301.0      62660
67000.0      56627
112824.0     52644
140451.0     50716
152576.0     49333
121476.0     48834
Name: count, dtype: int64
l14_inv_cross l14_inv_cross
116729.0    104981
66786.0      93420
20369.0      64855
25538.0      38859
120369.0     29545
85748.0      27672
121431.0     26149
59487.0      23723
91121.0      23570
43794.0      19825
Name: count, dtype: int64


In [21]:
# HASH_BUCKET = 200_000

# def hash_bucket_fast_str(arr_like, num_buckets: int) -> np.ndarray:
#     arr = pd.Series(arr_like, dtype="string").to_numpy()
#     return tf.strings.to_hash_bucket_fast(arr, num_buckets).numpy().astype("int32")

# def make_hash_with_unk(series: pd.Series, num_buckets: int) -> np.ndarray:
#     s = series.astype("string")
#     is_na = s.isna().to_numpy()

#     # NaN을 임시 토큰으로 채워 해시, 이후 +1 오프셋
#     hashed = hash_bucket_fast_str(s.fillna("__NA__"), num_buckets) + 1
#     hashed[is_na] = 0
#     return hashed.astype("int32")


In [22]:
# df["l_feat_14_hash"] = 0  # 기본값
# if "l_feat_14" in df.columns:
#     df.loc[tr_mask, "l_feat_14_hash"] = make_hash_with_unk(df.loc[tr_mask, "l_feat_14"], HASH_BUCKET)
#     df.loc[te_mask, "l_feat_14_hash"] = make_hash_with_unk(df.loc[te_mask, "l_feat_14"], HASH_BUCKET)
#     df.loc[va_mask, "l_feat_14_hash"] = make_hash_with_unk(df.loc[va_mask, "l_feat_14"], HASH_BUCKET)


*hour* , *day_of_week*
>

    dense(sin/cos)는 0부터 시작(0-based)

    sparse(카테고리)는 +1 해서 1부터(1-based), 0은 UNK

*age_group*
>

    (1) 임퓨팅 소스 불일치, (2) train-의존 매핑, (3) float 키 매칭, (4) 과적합.

In [23]:
def make_dense_sparse(
    _df,
    mask,
    *,
    ord_fit_cols,             # fit에 사용한 순서형 리스트  (예: ["hour","day_of_week","age_group"])
    cnt_fit_cols,             # fit에 사용한 순서형 리스트2 (  :  "l_feat_3","l_feat_27","feat_e_4" 등)
    real_fit_cols,            # 찐 연속형들
    all_dense_cols,           # ord_fit_cols + cnt_fit_cols + real_fit_cols
    imp_ord,imp_cnt,imp_real, # SimpleImputer(most_frequent/median), train으로만 fit된 것
    scaler,                   # MinMaxScaler, train으로만 fit된 것
    map_cols, mp_cont,        # 저카디널 연속형만 정수 매핑 정보(둘 다 train 기준)
    mp_age, mp_inv
):
    # --- 1) 같은 컬럼/순서로 transform (fit과 100% 동일) ---
    Xord  = imp_ord.transform(_df.loc[mask, ord_fit_cols])
    Xcnt  = imp_cnt.transform(_df.loc[mask, cnt_fit_cols])
    Xreal = imp_real.transform(_df.loc[mask,real_fit_cols ])

    # 기준값
    Xnum  = np.hstack([Xord, Xcnt, Xreal]).astype(np.float32)
    Xnum_s = scaler.transform(Xnum).astype(np.float32)

    # dense 초기화
    dense = pd.DataFrame(Xnum_s, columns= all_dense_cols, index=_df.index[mask])

    idx   = _df.index[mask]
    h_idx = ord_fit_cols.index("hour")
    d_idx = ord_fit_cols.index("day_of_week")


    # 임퓨팅 결과에서 바로 꺼냄
    hour_raw = pd.Series(Xord[:, h_idx], index=idx)
    dow_raw  = pd.Series(Xord[:, d_idx], index=idx)

    # 안전 캐스팅(소수 흔들림 방지)
    hour_raw = np.rint(hour_raw).astype("int32")
    dow_raw  = np.rint(dow_raw).astype("int32")

    # 0-based 정규화 (원본이 1..주기 or 0..주기 섞여 있어도 안전)
    def to_zero_based(x, mod):
        # 1..mod 형태면 -1, 그 외엔 모듈러로 강제 0..mod-1
        if x.min() >= 1 and x.max() <= mod:
            return (x - 1) % mod
        return x % mod

    hour0 = to_zero_based(hour_raw, 24)  # 0..23
    dow0  = to_zero_based(dow_raw, 7)    # 0..6

    # dense: 0-based로 주기형 계산
    dense["hour_sin"] = np.sin(2*np.pi * (hour0 / 24.0))
    dense["hour_cos"] = np.cos(2*np.pi * (hour0 / 24.0))
    dense["dow_sin"]  = np.sin(2*np.pi * (dow0  / 7.0))
    dense["dow_cos"]  = np.cos(2*np.pi * (dow0  / 7.0))

    dense_cols = all_dense_cols + ["hour_sin","hour_cos","dow_sin","dow_cos"]

    # sparse: +1 해서 1..주기, 0은 UNK로 예약
    sparse = pd.DataFrame(index=idx)
    sparse["hour_cat"]        = (hour0 + 1).astype("int32")        # 1..24
    sparse["day_of_week_cat"] = (dow0  + 1).astype("int32")        # 1..7





    # sparse 초기화
    sparse = pd.DataFrame(index=_df.index[mask])

    sparse["hour_cat"]        = (hour0 + 1).astype("int32")        # 1..24 (0=UNK)
    sparse["day_of_week_cat"] = (dow0  + 1).astype("int32")

    if "gender" in _df.columns:
        g = _df.loc[mask, "gender"].astype("float64")
        sparse["gender"] = g.where(g.isin([1, 2]), np.nan).fillna(0).astype("int32")

    idx   = _df.index[mask]
    a_idx = ord_fit_cols.index("age_group")
    # 임퓨팅된 age_group 값 → 정수로 확정
    age_imp = pd.Series(Xord[:, a_idx], index=idx).round().astype("int32")

    # 도메인 밖(이상값)은 0(UNK), 도메인 값은 그대로 사용
    sparse["age_group_cat"] = age_imp.where(age_imp.isin(AGE_DOMAIN), 0).astype("int32")
    sparse["inventory_id_cat"] = _df.loc[mask,"inventory_id"].map(mp_inv).fillna(0).astype("int32")


    # # 프리 해시
    # has_l14 = False
    # if "l_feat_14_hash" in _df.columns:
    #     sparse["l_feat_14_hash"] = _df.loc[mask, "l_feat_14_hash"].astype("int32")
    #     has_l14 = True
    # else: # 혹시나 프리 컴퓨트를 못한 경우 방지
    #     if "l_feat_14" in _df.columns:
    #       sparse["l_feat_14_hash"] = make_hash_with_unk(_df.loc[mask, "l_feat_14"], HASH_BUCKET)
    #     else:
    #       sparse["l_feat_14_hash"] = 0
    sparse["l14_hash"] = df.loc[mask, "l14_hash"].astype("int32")
    sparse["l14_inv_cross"] = df.loc[mask, "l14_inv_cross"].astype("int32")




    # 저카디널 연속형만 정수 매핑(임퓨트된 값 기준)
    idx = _df.index[mask]

    # 이미 있음: Xcnt = imp_cnt.transform(_df.loc[mask, cnt_fit_cols])  여기서 cnt_fit_cols = cont_cols
    for c in map_cols:  # = cont_cols
        j = _idx[c]  # cont_cols 기준 인덱스
        v = pd.Series(_round_vals(Xcnt[:, j]), index=idx)
        sparse[f"{c}_cat"] = v.map(mp_cont[c]).fillna(0).astype("int32")  # OOV/불일치 → 0(UNK)




    # 최종 sparse_cols 조립
    base_sparse = ["hour_cat","day_of_week_cat","age_group_cat","inventory_id_cat" , "l14_hash" , "l14_inv_cross"]
    if "gender" in sparse.columns:
        base_sparse = ["gender"] + base_sparse
    # if has_l14:
    #     base_sparse.append("l_feat_14_hash")

    sparse_cols = base_sparse + [c + "_cat" for c in map_cols]

    return dense[dense_cols], sparse[sparse_cols], dense_cols, sparse_cols



ORD_FIT_COLS  = ["hour","day_of_week","age_group"]   # inventory_id, l_feat_14는 제외(=sparse 전용)

ban = set(ORD_FIT_COLS + cont_cols + ["gender","inventory_id","l_feat_14","l14_hash","l14_inv_cross",
                                      seq_col, label_col, seq_len_col, "is_train", "ID"])

extra_cont_cols = [c for c in df.select_dtypes(include=["number"]).columns if c not in ban]


CNT_FIT_COLS   = cont_cols
REAL_FIT_COLS  = extra_cont_cols
ALL_DENSE_COLS = ORD_FIT_COLS + CNT_FIT_COLS + REAL_FIT_COLS




dense_tr, sparse_tr, dense_cols, sparse_cols = make_dense_sparse(
    df, tr_mask,
    ord_fit_cols=ORD_FIT_COLS,
    cnt_fit_cols=CNT_FIT_COLS,
    real_fit_cols=REAL_FIT_COLS,
    all_dense_cols=ALL_DENSE_COLS,
    imp_ord=imp_ord, imp_cnt=imp_cnt, imp_real=imp_real,
    scaler=scaler,
    map_cols=map_cols, mp_cont=mp_cont,
    mp_age=mp_age, mp_inv=mp_inv
)

dense_te, sparse_te, _, _ = make_dense_sparse(
    df, te_mask,
    ord_fit_cols=ORD_FIT_COLS,
    cnt_fit_cols=CNT_FIT_COLS,
    real_fit_cols=REAL_FIT_COLS,
    all_dense_cols=ALL_DENSE_COLS,
    imp_ord=imp_ord, imp_cnt=imp_cnt, imp_real=imp_real,
    scaler=scaler,
    map_cols=map_cols, mp_cont=mp_cont,
    mp_age=mp_age, mp_inv=mp_inv
)

dense_va, sparse_va, _, _ = make_dense_sparse(
    df, va_mask,
    ord_fit_cols=ORD_FIT_COLS,
    cnt_fit_cols=CNT_FIT_COLS,
    real_fit_cols=REAL_FIT_COLS,
    all_dense_cols=ALL_DENSE_COLS,
    imp_ord=imp_ord, imp_cnt=imp_cnt, imp_real=imp_real,
    scaler=scaler,
    map_cols=map_cols, mp_cont=mp_cont,
    mp_age=mp_age, mp_inv=mp_inv
)

# deepFM 컬럼 정의
다른 임베딩 차원 - 분리해야함


In [24]:
# HASH_BUCKET = 200_000
MAX_LEN = 150
EMBED_DIM = 4 # 과적합 방지

EMB_L14 = 8       # l14_hash embedding dim (보통 8~16)
EMB_CROSS = 16    # cross feature embedding dim (보통 16~32)



sparse_fixed = [
    SparseFeat('gender',            vocabulary_size=3,                  embedding_dim=EMBED_DIM, dtype='int32'),
    SparseFeat('inventory_id_cat',  vocabulary_size= inv_vocab_size,    embedding_dim= 8,       dtype='int32' , group_name= 'inv_8'),
    SparseFeat('hour_cat',          vocabulary_size=25,                 embedding_dim=EMBED_DIM, dtype='int32'),
    SparseFeat('day_of_week_cat',   vocabulary_size=8,                  embedding_dim=EMBED_DIM, dtype='int32'),
    SparseFeat('age_group_cat',     vocabulary_size= age_vocab_size,    embedding_dim=EMBED_DIM, dtype='int32'),
    SparseFeat("l14_hash",          vocabulary_size=art.l14_buckets,    embedding_dim=EMB_L14  , dtype='int32' , group_name= 'l14_hash'),
    SparseFeat("l14_inv_cross",     vocabulary_size=art.cross_buckets,  embedding_dim=EMB_CROSS, dtype='int32' , group_name= 'inv_cross_16')
]

# sparse_hash = [
#     SparseFeat('l_feat_14_hash', vocabulary_size= HASH_BUCKET + 1, embedding_dim= 16 , use_hash= False ,dtype='int32'),
# ]


dense_feats = [DenseFeat(c, 1) for c in dense_cols]



# 순서형(cont_cols) - sparse에 넣기
for c in map_cols:  # = cont_cols
    name = f"{c}_cat"
    if name in sparse_tr.columns:
        sparse_fixed.append(
            SparseFeat(name, vocabulary_size=vocab_cont[c], embedding_dim=EMBED_DIM, dtype='int32') # vocab_cont 위에서 UNK=0 예약 ㅇㅋ
        )


varlen_seq  = VarLenSparseFeat(
    sparsefeat = SparseFeat('seq' ,
                            vocabulary_size = vocab_seq , # train 기준으로 vocab 계산(0=패딩/UNK 전제)
                            embedding_dim   = 16 ,
                            # use_hash=True ,
                            dtype= 'int32', # 정수형 사용 -> 해시 안됨
                            group_name= 'seq_16'
                              ),
    maxlen   = MAX_LEN,
    combiner = 'mean',
    length_name = 'seq_len',
    weight_name = None,
    weight_norm = False
)




In [25]:
fixlen_feature_columns = sparse_fixed  + dense_feats #sparse_hash

varlen_feature_columns = [varlen_seq]


dnn_feature_columns = fixlen_feature_columns + varlen_feature_columns
linear_feature_columns = fixlen_feature_columns + varlen_feature_columns

feature_names = get_feature_names(dnn_feature_columns)

# DIN: 고정길이 + 시퀀스(VarLenSparseFeat) 함께 사용

# dnn_feature_columns_din = linear_feature_columns + [varlen_seq]
# behavior_feature_list = ['inventory_id']  # query(현재 타깃)와 history 키 그룹 매칭


## 0=패딩/UNK 전역 규칙 검사

In [26]:
def check_sparse_df(df_sparse, vocab_size_map, zero_is_unk_map=None, warn_unk_thresh=0.05):
    """
    df_sparse        : pd.DataFrame (sparse_tr / sparse_te / sparse_va 중 하나)
    vocab_size_map   : {col: vocab_size}  # 예: {'hour_cat':25, 'age_group_cat': age_vocab_size, ...}
    zero_is_unk_map  : {col: bool}        # True면 0=UNK, False면 0은 정상 토큰
    warn_unk_thresh  : UNK 비율 경고 임계값(기본 5%)
    """
    if zero_is_unk_map is None:
        # 기본은 전부 0=UNK로 간주
        zero_is_unk_map = {c: True for c in vocab_size_map.keys()}

    fails = []
    report_lines = []

    for col, vsz in vocab_size_map.items():
        if col not in df_sparse.columns:
            report_lines.append(f"[SKIP] {col}: column missing in DataFrame")
            continue

        s = df_sparse[col].to_numpy()

        # 1) 정수형 체크
        if not np.issubdtype(s.dtype, np.integer):
            fails.append(f"[{col}] dtype={s.dtype} (must be integer)")

        # 2) 값 범위 체크
        minv = int(np.nanmin(s)) if s.size else 0
        maxv = int(np.nanmax(s)) if s.size else 0

        if minv < 0:
            fails.append(f"[{col}] has negative indices (min={minv})")
        if maxv >= vsz:
            fails.append(f"[{col}] value {maxv} >= vocab_size {vsz}")

        # 3) UNK(0) 비율 리포트/경고
        zero_is_unk = zero_is_unk_map.get(col, True)
        unk_rate = float((s == 0).mean()) if s.size else 0.0

        note = ""
        if zero_is_unk:
            if unk_rate > warn_unk_thresh:
                note = f" [WARN: UNK rate {unk_rate:.2%} > {warn_unk_thresh:.0%}]"
        else:
            note = " [NOTE: 0 is a normal token here]"

        report_lines.append(
            f"[{col}] ok: min={minv}, max={maxv}, vocab={vsz}, UNK_rate={unk_rate:.2%}{note}"
        )

    if fails:
        msg = "Sparse checks FAILED:\n" + "\n".join(fails)
        raise AssertionError(msg)

    return "\n".join(report_lines)


In [27]:
# 1) vocab 크기 사전
vocab_size_map = {
    "gender": 3,                       # 0=UNK, 1,2
    "inventory_id_cat": inv_vocab_size,
    "hour_cat": 25,                    # 0..24
    "day_of_week_cat": 8,              # 0..7
    "age_group_cat": age_vocab_size,
    "l14_hash": art.l14_buckets,
    "l14_inv_cross" : art.cross_buckets,
    **{f"{c}_cat": (len(mp_cont[c]) + 1) for c in map_cols}  # 0=UNK 포함
}

# 2) 0이 UNK인지 여부 (해시에만 예외 둘 수 있음)
zero_is_unk_map = {col: True for col in vocab_size_map}


# 3) split별 점검
print("=== TRAIN ===")
print(check_sparse_df(sparse_tr, vocab_size_map, zero_is_unk_map))
print("=== VALID ===")
print(check_sparse_df(sparse_va, vocab_size_map, zero_is_unk_map))
print("=== TEST ===")
print(check_sparse_df(sparse_te, vocab_size_map, zero_is_unk_map))


=== TRAIN ===
[gender] ok: min=1, max=2, vocab=3, UNK_rate=0.00%
[inventory_id_cat] ok: min=0, max=0, vocab=10, UNK_rate=100.00% [WARN: UNK rate 100.00% > 5%]
[hour_cat] ok: min=1, max=24, vocab=25, UNK_rate=0.00%
[day_of_week_cat] ok: min=1, max=7, vocab=8, UNK_rate=0.00%
[age_group_cat] ok: min=1, max=8, vocab=9, UNK_rate=0.00%
[l14_hash] ok: min=0, max=198547, vocab=200000, UNK_rate=10.00% [WARN: UNK rate 10.00% > 5%]
[l14_inv_cross] ok: min=8, max=131052, vocab=131072, UNK_rate=0.00%
[l_feat_3_cat] ok: min=1, max=3, vocab=4, UNK_rate=0.00%
[l_feat_27_cat] ok: min=1, max=5, vocab=6, UNK_rate=0.00%
[feat_e_4_cat] ok: min=1, max=1, vocab=2, UNK_rate=0.00%
[feat_a_1_cat] ok: min=1, max=5, vocab=6, UNK_rate=0.00%
[feat_a_3_cat] ok: min=1, max=6, vocab=7, UNK_rate=0.00%
[feat_a_4_cat] ok: min=1, max=6, vocab=7, UNK_rate=0.00%
[feat_a_6_cat] ok: min=1, max=10, vocab=11, UNK_rate=0.00%
[feat_a_7_cat] ok: min=1, max=9, vocab=10, UNK_rate=0.00%
[feat_a_8_cat] ok: min=1, max=7, vocab=8, UNK_r

In [28]:
# 값 범위 및 UNK 비율 빠른 점검
def quick_check(s, vocab_size, name):
    mn, mx = int(s.min()), int(s.max())
    unk_rate = float((s == 0).mean())
    print(f"{name:16s} range=[{mn},{mx}] vs vocab={vocab_size} | UNK={unk_rate:.3f}")

quick_check(df.loc[tr_mask, "l14_hash"],      art.l14_buckets,   "l14_hash(tr)")
quick_check(df.loc[va_mask, "l14_hash"],      art.l14_buckets,   "l14_hash(va)")
quick_check(df.loc[te_mask, "l14_hash"],      art.l14_buckets,   "l14_hash(te)")
quick_check(df.loc[tr_mask, "l14_inv_cross"], art.cross_buckets, "l14_cross(tr)")
quick_check(df.loc[va_mask, "l14_inv_cross"], art.cross_buckets, "l14_cross(va)")
quick_check(df.loc[te_mask, "l14_inv_cross"], art.cross_buckets, "l14_cross(te)")


l14_hash(tr)     range=[0,198547] vs vocab=200000 | UNK=0.100
l14_hash(va)     range=[0,198547] vs vocab=200000 | UNK=0.101
l14_hash(te)     range=[0,198547] vs vocab=200000 | UNK=0.313
l14_cross(tr)    range=[8,131052] vs vocab=131072 | UNK=0.000
l14_cross(va)    range=[8,131052] vs vocab=131072 | UNK=0.000
l14_cross(te)    range=[8,131052] vs vocab=131072 | UNK=0.000


In [29]:
# s = df["l_feat_14_hash"].to_numpy()
# assert s.min() >= 0
# assert s.max() <= HASH_BUCKET  # 1..HASH_BUCKET 또는 0
# # zero_is_unk_map["l_feat_14_hash"] = True 로 체크 함수와 정책 일치


In [30]:
# # 검증용
# assert seq_tr_pad.max() < vocab_seq and seq_te_pad.max() < vocab_seq

# 학습 샘플 생성 및 모델 학습

DeepCTR 모델은 내부적으로 특성 이름별로 Input Layer를 자동 생성한다.

그래서 입력을 dict 형태로 요구한다.


<br>
주의
>

    DataFrame의 seq는 건들지 말고, 모델에 넣을 때만 seq_padded/seq_len을 넘긴다


In [31]:
def to_inputs_fast(dense_df, sparse_df, dense_cols, sparse_cols, *, seq_pad=None, seq_len=None):

    X = {}


    # Sparse: 정수 인덱스 vs 문자열 해시 구분
    for c in sparse_cols:
        s = sparse_df[c]
        if str(s.dtype).startswith("string") or s.dtype == object:
            # DeepCTR 해시용: str 배열
            X[c] = s.astype("string").to_numpy(dtype=object, copy=False)
        else:
            # 프리 해시
            X[c] = s.to_numpy(dtype=np.int32, copy=False)


    # Dense: float32
    for c in dense_cols:
        X[c] = dense_df[c].to_numpy(dtype=np.float32, copy=False)


    # VarLen
    if seq_pad is not None:
        X["seq"]     = np.asarray(seq_pad, dtype=np.int32)
    if seq_len is not None:
        X["seq_len"] = np.asarray(seq_len, dtype=np.int32)
    return X




X_train = to_inputs_fast(dense_tr, sparse_tr, dense_cols, sparse_cols,
                         seq_pad=seq_tr_pad if 'seq' in feature_names else None,
                         seq_len=seq_tr_len if 'seq' in feature_names else None)
X_test  = to_inputs_fast(dense_te, sparse_te, dense_cols, sparse_cols,
                         seq_pad=seq_te_pad if 'seq' in feature_names else None,
                         seq_len=seq_te_len if 'seq' in feature_names else None)
X_valid = to_inputs_fast(dense_va, sparse_va, dense_cols, sparse_cols,
                         seq_pad=seq_va_pad if 'seq' in feature_names else None,
                         seq_len=seq_va_len if 'seq' in feature_names else None)



train_model_input = X_train
test_model_input  = X_test
valid_model_input = X_valid
train_y = df.loc[tr_mask, "clicked"].astype("float32").to_numpy()


In [32]:
seq_tr_pad, seq_tr_len, vocab_seq = build_seq_padded_len(
    df.loc[tr_mask, "seq"], MAX_LEN, offset=1, ignore_neg=True
)

seq_te_pad, seq_te_len, _ = build_seq_padded_len(
    df.loc[te_mask, "seq"], MAX_LEN, offset=1, ignore_neg=True
)

seq_va_pad, seq_va_len, _ = build_seq_padded_len(
    df.loc[va_mask, "seq"], MAX_LEN , offset=1, ignore_neg=True
)




# train vocab 기준으로 초과 인덱스는 0(PAD/UNK) 처리
seq_tr_pad = np.where(seq_tr_pad < vocab_seq, seq_tr_pad, 0).astype("int32")
seq_te_pad = np.where(seq_te_pad < vocab_seq, seq_te_pad, 0).astype("int32")
seq_va_pad = np.where(seq_va_pad < vocab_seq, seq_va_pad, 0).astype("int32")


seq_tr_len = seq_tr_len.astype("int32"); seq_te_len = seq_te_len.astype("int32")

# 1) 편의 이름 집합
# hash_names  = [f.name for f in sparse_hash]     # 해시(문자열) SparseFeat들
fixed_names = [f.name for f in sparse_fixed]    # 정수 인덱스 SparseFeat들
dense_names = [f.name for f in dense_feats]

In [33]:
target = 'clicked'


train_model_input = {}
test_model_input  = {}
valid_model_input = {}

for name in feature_names:
    if name == "seq":
        train_model_input[name] = np.asarray(seq_tr_pad, dtype=np.int32)
        test_model_input[name]  = np.asarray(seq_te_pad, dtype=np.int32)
        valid_model_input[name] = np.asarray(seq_va_pad, dtype=np.int32)

    elif name == "seq_len":
        train_model_input[name] = np.asarray(seq_tr_len, dtype=np.int32)
        test_model_input[name]  = np.asarray(seq_te_len, dtype=np.int32)
        valid_model_input[name] = np.asarray(seq_va_len, dtype=np.int32)

    # elif name in hash_names:
    #     train_model_input[name] = sparse_tr[name].to_numpy(dtype=np.int32)
    #     test_model_input[name]  = sparse_te[name].to_numpy(dtype=np.int32)
    #     valid_model_input[name] = sparse_va[name].to_numpy(dtype=np.int32)

    elif name in fixed_names:
        train_model_input[name] = sparse_tr[name].to_numpy(dtype=np.int32)
        test_model_input[name]  = sparse_te[name].to_numpy(dtype=np.int32)
        valid_model_input[name] = sparse_va[name].to_numpy(dtype=np.int32)

    else:  # DenseFeat
        train_model_input[name] = dense_tr[name].to_numpy(dtype=np.float32)
        test_model_input[name]  = dense_te[name].to_numpy(dtype=np.float32)
        valid_model_input[name] = dense_va[name].to_numpy(dtype=np.float32)

train_y = df.loc[tr_mask, target].astype("float32").to_numpy()

In [34]:
# 학습 타깃 (train 구간 전부)
y_tr = train_y.astype(int)
pos = (y_tr==1).sum()
neg = (y_tr==0).sum()
w1 = 0.5 / max(pos,1) # 양성 총합이 0.5
w0 = 0.5 / max(neg,1) # 음성 총합이 0.5
sw = np.where(y_tr==1, w1, w0).astype("float32")


y_va = df.loc[va_mask, "clicked"].astype(int).to_numpy()


## ASHA
> 리소스(에폭 수 등)를 단계적으로 늘려가며, 각 단계(rung)에서 상위 1/reduction_factor 비율만 다음 단계로 승격.

In [35]:
# import numpy as np, optuna
# from optuna.samplers import TPESampler
# from optuna.pruners import SuccessiveHalvingPruner
# from sklearn.metrics import roc_auc_score
# from deepctr.models import DeepFM

# SEED = 42
# np.random.seed(SEED)

# # ---- 하이퍼공간(딥CTR-TF에 맞춤) ----
# HIDDEN_CHOICES = [
#     (512, 256, 128),
#     (256, 128, 64),
#     (384, 192, 96),
# ]

# def build_model(params):
#     # DeepCTR(TF/Keras) 버전 — device 인자 없음!
#     model = DeepFM(
#         linear_feature_columns=linear_feature_columns,
#         dnn_feature_columns=dnn_feature_columns,
#         task='binary',
#         l2_reg_embedding=params["l2_reg_embedding"],
#         l2_reg_linear=params["l2_reg_linear"],
#         dnn_hidden_units=params["dnn_hidden_units"],
#         dnn_dropout=params["dnn_dropout"]
#     )
#     model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["AUC"])
#     return model

# def objective(trial: optuna.Trial):
#     params = {
#         "l2_reg_embedding": trial.suggest_float("l2_reg_embedding", 1e-6, 1e-4, log=True),
#         "l2_reg_linear":    trial.suggest_float("l2_reg_linear",    1e-7, 1e-5, log=True),
#         "dnn_dropout":      trial.suggest_float("dnn_dropout",      0.1, 0.5),
#         # tuple로 바꿔 경고 제거
#         "dnn_hidden_units": trial.suggest_categorical("dnn_hidden_units", HIDDEN_CHOICES),
#         "batch_size":       trial.suggest_categorical("batch_size", [512, 1024, 2048]),
#         "epochs":           10,
#     }

#     model = build_model(params)

#     best_auc = 0.0
#     max_epochs = params["epochs"]
#     bs = params["batch_size"]

#     for ep in range(1, max_epochs + 1):
#         model.fit(
#             x=train_model_input,
#             y=y_tr,
#             sample_weight=sw,              # train에만 가중치
#             batch_size=bs,
#             epochs=1,
#             verbose=0,
#             validation_data=(valid_model_input, y_va),
#         )
#         # 중간 평가 → 프루닝 판단
#         y_pred = model.predict(valid_model_input, batch_size=4096, verbose=0)
#         auc = float(roc_auc_score(y_va, y_pred))
#         best_auc = max(best_auc, auc)

#         trial.report(auc, step=ep)
#         if trial.should_prune():
#             raise optuna.TrialPruned()

#     return best_auc

# # 실험 경고가 싫다면 multivariate/group 사용 안 함
# sampler = TPESampler(seed=SEED)  # multivariate=False, group=False (default)

# pruner = SuccessiveHalvingPruner(
#     min_resource=2,        # 에폭 2부터 비교
#     reduction_factor=3,    # 상위 1/3만 다음 라운드
#     min_early_stopping_rate=0
# )

# study = optuna.create_study(
#     direction="maximize",
#     sampler=sampler,
#     pruner=pruner,
#     study_name="deepfm_asha"
# )

# study.optimize(objective, n_trials=40, n_jobs=1, show_progress_bar=True)

# print("Best AUC:", study.best_value)
# print("Best params:", study.best_params)
# best_params = study.best_params


In [36]:
# model = DeepFM(linear_feature_columns, dnn_feature_columns, task='binary')

# model.compile("adam", "binary_crossentropy",
#               metrics=['binary_crossentropy'], )

In [37]:
# history = model.fit(
#     train_model_input,
#     y_tr,
#     sample_weight=sw,
#     validation_data=(valid_model_input, y_va),
#     batch_size = 512,
#     epochs = 7,
#     verbose = 2
# )


In [38]:
# # 제출용 저장
# pred_ans = model.predict(test_model_input, batch_size=512)

# 보정기: OOF로 학습

Inventory 단위 그룹 KFold면 충분한가?
>

    Inventory는 “노출 맥락(슬롯/지면)”일 가능성이 높고, 실제 서빙에서 바뀌기 쉬운 축
> 누수 방지에는 기여했지만, 서빙 분포 근사와 일반화 위험(특히 item/sequence 축)을 충분히 커버했다고 단정하긴 어렵다.

----
> inventroy_id / l_feat_14를 그룹으로 묶은 다음 day_of_week를 1-6까지 학습 , 이후 7만 검증 데이터셋에 넣고 학습 -> inventroy_id x l_feat_14 교차해시로 하면 근소하게 성능 향상함

In [39]:
# ==== HARD-CODED PARAMS ====
L2_EMB   = 3e-6
L2_LIN   = 1e-6
DROPOUT  = 0.2
HIDDEN   = (256, 128)
LR       = 1e-3
BS_TRAIN = 512
EPOCHS   = 7
VAL_FREQ = 1


In [40]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score, log_loss

import tensorflow as tf
from deepctr.feature_column import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr.models import DeepFM

## CV-A (튜닝/선정용, 안정성): inventory_id ⊕ l_feat_14

복합 그룹을 사용한 StratifiedGroupKFold, day_of_week ∈ {1..6}만으로 교차검증(=실제 학습 가능한 세계).

>

    inventory_id + l_feat_14 조합을 그룹으로 묶고 day_of_week이 1~6인 데이터만을 사용해서 StratifiedGroupKFold로 교차 검증을 수행한다.

In [41]:
X_tr_all = train_model_input        # train 구간 전체 입력 (dict of np arrays)
y_tr_all = y_tr.astype(int)
sw_all   = sw
df_tr    = df.loc[tr_mask].copy()


# ----그룹 / 층화 기준

inv = df['inventory_id'].to_numpy()
ad_14 = df['l_feat_14'].to_numpy()
y = y_tr_all

#-----그룹 복합키
grp_tuple = list(zip(inv, ad_14))
grp_code, _ = pd.factorize(grp_tuple)     # 0..G-1 | GroupKFold의 groups 파라미터로 들어감.

# 블록 마스크: day_of_week

dow   = df_tr["day_of_week"].to_numpy()
mask7 = (dow == 7)
mask1_6 = ~mask7


/tmp/ipython-input-41-1874194580.py:15: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  grp_code, _ = pd.factorize(grp_tuple)     # 0..G-1 | GroupKFold의 groups 파라미터로 들어감.


In [42]:
cv = StratifiedGroupKFold(n_splits = 5 , shuffle= True , random_state= 42)

idx_1_6   = np.where(mask1_6)[0] # day 1~6 행 인덱스
y_1_6     = y[idx_1_6]           # day 1~6의 라벨값
grp_1_6 = grp_code[idx_1_6]    # inventroy_id + l_feat_14 그룹에서 day_of_week가 1-6인 값만 추출


# DeepCTR 입력이 dict -> fold별로 인덱싱이 가능하도록 헬퍼 준비
def slice_X_dict(X_dict, idx):
    return {k: v[idx] for k, v in X_dict.items()}


oof_pred = np.full_like(y, fill_value=np.nan, dtype=np.float32)

fold_models = []           # fold 모델을 저장해 CV-B에서 앙상블 예측에 쓸 수 있음
fold_metrics = []          # 각 fold의 OOF 지표 저장



## CV-B (서빙 근사용, 위험 점검)

>
    day_of_week=7 전부를 고정 홀드아웃으로 두고, 각 fold에서 학습된 모델로 항상 별도 평가.

In [43]:
import gc

In [44]:
def build_deepfm():
    model = DeepFM(
        linear_feature_columns=linear_feature_columns,
        dnn_feature_columns=dnn_feature_columns,
        task='binary',
        l2_reg_embedding=L2_EMB,
        l2_reg_linear=L2_LIN,
        dnn_hidden_units=HIDDEN,
        dnn_dropout=DROPOUT,
        dnn_use_bn=True
    )
    opt = tf.keras.optimizers.Adam(learning_rate=LR)

    model.compile(
        optimizer=opt,
        loss="binary_crossentropy",
        metrics = [],                                         # 주 지표는 val_loss
        weighted_metrics=[tf.keras.metrics.AUC(name="auc")]   # 로그용
    )
    return model

In [45]:
# 모델 학습
for fold, (tr_idx_rel, va_idx_rel) in enumerate(cv.split(X=grp_1_6, y=y_1_6, groups=grp_1_6), 1):
    print(f"===== Fold {fold} =====")
    tr_idx = idx_1_6[tr_idx_rel]
    va_idx = idx_1_6[va_idx_rel]

    X_tr = slice_X_dict(X_tr_all, tr_idx)
    X_va = slice_X_dict(X_tr_all, va_idx)
    y_tr = y[tr_idx]
    y_va = y[va_idx]
    sw_tr = None if sw_all is None else sw_all[tr_idx]
    sw_va = None if sw_all is None else sw_all[va_idx]


    tf.keras.backend.clear_session()  # 매 fold마다 세션 정리
    gc.collect()                      # 학습 로그 저장

    model_fold = build_deepfm()       # 매 폴드 새로 생성·컴파일

    callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min", patience=2, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", mode="min", factor=0.5, patience=2, min_lr=1e-6
    ),
]


    model_fold.fit(
        x=X_tr, y=y_tr,
        sample_weight=sw_tr,
        validation_data=(X_va, y_va, sw_va),   # 검증 가중치 그대로 사용
        batch_size=BS_TRAIN,
        epochs=EPOCHS,
        verbose=2,
        callbacks=callbacks,
        validation_freq=VAL_FREQ,
    )



# OOF 예측
oof_pred[va_idx] = model_fold.predict(X_va).reshape(-1)

#  fold 모델 보관 -> 블록7 평가 때 앙상블
fold_models.append(model_fold)

===== Fold 1 =====
Epoch 1/7
4507/4507 - 72s - loss: 2.9094e-07 - auc: 0.5530 - val_loss: 2.1006e-07 - val_auc: 0.5851 - lr: 0.0010 - 72s/epoch - 16ms/step
Epoch 2/7
4507/4507 - 66s - loss: 2.1477e-07 - auc: 0.5875 - val_loss: 2.0853e-07 - val_auc: 0.5965 - lr: 0.0010 - 66s/epoch - 15ms/step
Epoch 3/7
4507/4507 - 66s - loss: 2.1309e-07 - auc: 0.5969 - val_loss: 2.0831e-07 - val_auc: 0.6019 - lr: 0.0010 - 66s/epoch - 15ms/step
Epoch 4/7
4507/4507 - 67s - loss: 2.1228e-07 - auc: 0.6037 - val_loss: 2.0805e-07 - val_auc: 0.6036 - lr: 5.0000e-04 - 67s/epoch - 15ms/step
Epoch 5/7
4507/4507 - 66s - loss: 2.1208e-07 - auc: 0.6056 - val_loss: 2.0791e-07 - val_auc: 0.6052 - lr: 5.0000e-04 - 66s/epoch - 15ms/step
Epoch 6/7
4507/4507 - 66s - loss: 2.1186e-07 - auc: 0.6068 - val_loss: 2.0782e-07 - val_auc: 0.6060 - lr: 2.5000e-04 - 66s/epoch - 15ms/step
Epoch 7/7
4507/4507 - 66s - loss: 2.1183e-07 - auc: 0.6074 - val_loss: 2.0773e-07 - val_auc: 0.6065 - lr: 2.5000e-04 - 66s/epoch - 15ms/step
===== 

oof 학습하기

In [46]:
from sklearn.linear_model import LogisticRegression

def platt_fit(p, y, w=None, eps=1e-12):
    p = np.clip(np.asarray(p), eps, 1-eps)
    z = np.log(p/(1-p))[:, None]                  # logit
    clf = LogisticRegression(max_iter=1000)      # log-loss를 직접 최소화
    clf.fit(z, y.astype(int), sample_weight=w)
    def calibrate(q):
        q = np.clip(np.asarray(q), eps, 1-eps)
        zq = np.log(q/(1-q))[:, None]
        return clf.predict_proba(zq)[:, 1]
    return calibrate

In [47]:
# ===== 0) 사전 준비: 마스크/그룹/OOF 컨테이너 =====
dow = df.loc[tr_mask, "day_of_week"].to_numpy()
idx_1_6 = np.where(dow != 7)[0]
idx_7   = np.where(dow == 7)[0]

y_1_6   = y_tr_all[idx_1_6]
grp_1_6 = grp_code[idx_1_6]   # (inventory_id, l_feat_14) factorize한 값


In [48]:
# ===== 2) Platt: OOF(1~6 유효분)로만 학습 =====
valid_mask = ~np.isnan(oof_pred)
platt = platt_fit(oof_pred[valid_mask], y_tr_all[valid_mask], w=None if sw_all is None else sw_all[valid_mask])

# ===== 3) CV-B: day 7 고정 홀드아웃 구성 (최종 검증셋) =====
valid_model_input = slice_X_dict(X_tr_all, idx_7)
y_va = y_tr_all[idx_7]

# 검증(holdout) 가중치: 대회 산식 유지
pos_va = int((y_va == 1).sum())
neg_va = int((y_va == 0).sum())
w1_va = 0.5 / max(pos_va, 1)
w0_va = 0.5 / max(neg_va, 1)
sw_va = np.where(y_va == 1, w1_va, w0_va).astype("float32")

# ===== 4) 최종 모델: train 전체(=1~6)로 재학습 후 day 7 평가 =====
final_model = build_deepfm()
final_model.fit(
    x=slice_X_dict(X_tr_all, idx_1_6), y=y_tr_all[idx_1_6],
    sample_weight=None if sw_all is None else sw_all[idx_1_6],
    validation_data=(valid_model_input, y_va, sw_va),   # holdout 7 + 가중치
    batch_size=BS_TRAIN,
    epochs=EPOCHS,
    verbose=2,
    callbacks=callbacks,
    validation_freq=VAL_FREQ,
)

# ===== 5) 예측 + 보정 =====
p_valid_raw_7 = final_model.predict(valid_model_input, batch_size=1024, verbose=0).ravel().astype("float32")
p_valid_cal_7 = platt(p_valid_raw_7).astype("float32")

p_test_raw_7  = final_model.predict(test_model_input, batch_size=1024, verbose=0).ravel().astype("float32")
p_test_cal_7  = platt(p_test_raw_7).astype("float32")

Epoch 1/7
5581/5581 - 86s - loss: 2.8387e-07 - auc: 0.5598 - val_loss: 1.9393e-06 - val_auc: 0.6027 - lr: 0.0010 - 86s/epoch - 15ms/step
Epoch 2/7
5581/5581 - 81s - loss: 2.1313e-07 - auc: 0.5918 - val_loss: 1.9332e-06 - val_auc: 0.6107 - lr: 0.0010 - 81s/epoch - 15ms/step
Epoch 3/7
5581/5581 - 81s - loss: 2.1171e-07 - auc: 0.6016 - val_loss: 1.9308e-06 - val_auc: 0.6129 - lr: 0.0010 - 81s/epoch - 15ms/step
Epoch 4/7
5581/5581 - 81s - loss: 2.1123e-07 - auc: 0.6068 - val_loss: 1.9298e-06 - val_auc: 0.6135 - lr: 5.0000e-04 - 81s/epoch - 15ms/step
Epoch 5/7
5581/5581 - 81s - loss: 2.1097e-07 - auc: 0.6087 - val_loss: 1.9286e-06 - val_auc: 0.6145 - lr: 5.0000e-04 - 81s/epoch - 15ms/step
Epoch 6/7
5581/5581 - 81s - loss: 2.1070e-07 - auc: 0.6109 - val_loss: 1.9293e-06 - val_auc: 0.6154 - lr: 2.5000e-04 - 81s/epoch - 15ms/step
Epoch 7/7
5581/5581 - 81s - loss: 2.1068e-07 - auc: 0.6112 - val_loss: 1.9274e-06 - val_auc: 0.6157 - lr: 2.5000e-04 - 81s/epoch - 15ms/step


### 1) 블록7(holdout)에서 fold 앙상블 평가

In [49]:
# ===== Fold-ensemble on holdout(= day_of_week==7) =====
# valid_model_input, y_va, sw_va 가 이미 준비되어 있다고 가정
pred_7_stack = []
for m in fold_models:
    pred_7_stack.append(
        m.predict(valid_model_input, batch_size=1024, verbose=0).ravel()
    )

# 폴드 평균(간단하고 강력)
p_valid_raw_ens = np.mean(np.vstack(pred_7_stack), axis=0).astype("float32")
p_valid_cal_ens = platt(p_valid_raw_ens).astype("float32")   # ← OOF로 학습한 Platt 사용

# (선택) 지표 확인
from sklearn.metrics import log_loss, average_precision_score

logloss_valid_raw_ens = log_loss(y_va, p_valid_raw_ens, sample_weight=sw_va)
logloss_valid_cal_ens = log_loss(y_va, p_valid_cal_ens, sample_weight=sw_va)

prauc_valid_raw_ens   = average_precision_score(y_va, p_valid_raw_ens, sample_weight=sw_va)
prauc_valid_cal_ens   = average_precision_score(y_va, p_valid_cal_ens, sample_weight=sw_va)

print(
    f"[HOLDOUT-ENSEMBLE] logloss raw={logloss_valid_raw_ens:.6f} "
    f"cal={logloss_valid_cal_ens:.6f} | prAUC raw={prauc_valid_raw_ens:.6f} "
    f"cal={prauc_valid_cal_ens:.6f}"
)


[HOLDOUT-ENSEMBLE] logloss raw=0.670775 cal=0.693356 | prAUC raw=0.620857 cal=0.620854


### 2) 테스트 예측도 fold 앙상블로 진행

In [50]:
# ===== Fold-ensemble on test =====
pred_test_stack = []
for m in fold_models:
    pred_test_stack.append(
        m.predict(test_model_input, batch_size=1024, verbose=0).ravel()
    )

p_test_raw_ens = np.mean(np.vstack(pred_test_stack), axis=0).astype("float32")
p_test_cal_ens = platt(p_test_raw_ens).astype("float32")   # ← 동일 Platt 보정


# 퍼뮤테이션 중요도

In [57]:
from copy import deepcopy
from sklearn.metrics import log_loss, average_precision_score

def perm_importance(models, X_val, y_val, base_metric='logloss', n_rounds=3, metric_fn=None, batch_size=1024):
    # Calculate base score for each model and average
    base_scores = []
    for model in models:
        p = model.predict(X_val, batch_size=batch_size, verbose=0).ravel()
        if metric_fn is None:
            if base_metric == 'logloss':
                base_scores.append(log_loss(y_val, p))
            elif base_metric == 'prauc':
                base_scores.append(average_precision_score(y_val, p))
            else:
                raise ValueError("metric_fn or known base_metric required")
        else:
            base_scores.append(metric_fn(y_val, p))

    base = np.mean(base_scores)
    larger_is_better = base_metric == 'prauc' if metric_fn is None else False # Assuming metric_fn returns larger_is_better=False by default

    imp = {}
    for name, arr in X_val.items():
        vals = []
        for _ in range(n_rounds):
            Xp = deepcopy(X_val)
            perm = np.random.permutation(len(arr))
            Xp[name] = arr[perm]
            round_scores = []
            for model in models:
                pp = model.predict(Xp, batch_size=batch_size, verbose=0).ravel()
                score = log_loss(y_val, pp) if base_metric=='logloss' else average_precision_score(y_val, pp)
                round_scores.append(score)
            vals.append(np.mean(round_scores))
        score_mean = float(np.mean(vals))
        # 중요도: 성능 악화량
        delta = (score_mean - base) if not larger_is_better else (base - score_mean)
        imp[name] = delta
    # 큰 값일수록 중요
    return dict(sorted(imp.items(), key=lambda x: x[1], reverse=True))

imp_logloss = perm_importance(fold_models, valid_model_input, y_va, base_metric='logloss')
print(imp_logloss)

{'feat_c_5': 0.007724524516580122, 'l_feat_16': 0.0072490688118981295, 'feat_c_4': 0.006182657401823, 'hour_sin': 0.006139779068609852, 'feat_c_8': 0.005287805355955744, 'hour': 0.005151625326430209, 'age_group': 0.004395740110799573, 'l_feat_1': 0.0035720670711991698, 'l_feat_3': 0.003447566614321107, 'l_feat_13': 0.0032852387042897613, 'feat_a_6': 0.003070319162408408, 'feat_a_5': 0.0027569747721502003, 'feat_e_8': 0.00221322700115667, 'feat_a_7': 0.002213041426950668, 'l_feat_9': 0.0019045452692313392, 'feat_d_6': 0.0017495667371421053, 'l_feat_4': 0.0012000936895811831, 'feat_a_18': 0.0011707501501074802, 'l_feat_17': 0.001167813636842041, 'history_a_7': 0.0008164104822391272, 'feat_c_7': 0.0007591904737493937, 'hour_cos': 0.0007544827925590747, 'feat_a_9': 0.0006916749419890733, 'history_b_6': 0.0006220330141519792, 'feat_c_6': 0.0005976335345374562, 'feat_e_10': 0.0005822393461821784, 'feat_a_15': 0.0005371906346415267, 'feat_e_3': 0.0005361874942692024, 'feat_d_4': 0.00050108794

In [61]:
pd.set_option('display.max_rows', None)

imp_df = pd.DataFrame(list(imp_logloss.items()), columns=['feature', 'importance'])

display(imp_df)

,feature,importance
0,feat_c_5,7.724525e-03
1,l_feat_16,7.249069e-03
2,feat_c_4,6.182657e-03
3,hour_sin,6.139779e-03
4,feat_c_8,5.287805e-03
5,hour,5.151625e-03
6,age_group,4.395740e-03
7,l_feat_1,3.572067e-03
8,l_feat_3,3.447567e-03
9,l_feat_13,3.285239e-03


### 검증 분포에 재맞춤(더 타이트하게)

In [ ]:
# platt_va = platt_fit(p_valid_raw, y_va, w=sw_va)   # 검증에서 다시 맞춤

# p_valid_cal_ = platt_va(p_valid_raw).astype("float32")
# p_test_cal_  = platt_va(p_test_raw).astype("float32")


대회 산식

In [63]:
from sklearn.metrics import average_precision_score, log_loss

def weighted_logloss(y, p, eps=1e-12):
    y = np.asarray(y).astype(int)
    p = np.clip(np.asarray(p), eps, 1-eps)
    n1 = (y==1).sum(); n0 = (y==0).sum()
    w = np.where(y==1, 0.5/max(n1,1), 0.5/max(n0,1))
    return log_loss(y, p, sample_weight=w, labels=[0,1])

def lb_score(y, p):
    AP  = average_precision_score(y, p)
    WLL = weighted_logloss(y, p)
    return AP, WLL, 0.5*AP + 0.5*(1.0/(1.0+WLL))

# Evaluate on the validation set (day_of_week == 7)
AP_raw, WLL_raw, S_raw = lb_score(y_va, p_valid_raw_ens)
AP_cal, WLL_cal, S_cal = lb_score(y_va, p_valid_cal_ens)

print(f"Validation (Holdout 7) Scores:")
print(f"Raw: AP={AP_raw:.6f}, WLL={WLL_raw:.6f}, Score={S_raw:.6f}")
print(f"Calibrated: AP={AP_cal:.6f}, WLL={WLL_cal:.6f}, Score={S_cal:.6f}")


Validation (Holdout 7) Scores:
Raw: AP=0.032270, WLL=0.670775, Score=0.315397
Calibrated: AP=0.032269, WLL=0.693356, Score=0.311406


모델 저장

In [ ]:
# save_dir = "/content/drive/MyDrive/Colab Notebooks/CRT/model"
# import os
# os.makedirs(save_dir, exist_ok=True)

# # 저장: 체크포인트 형식(.index, .data-00000-of-00001 파일 세트)
# ckpt_path = f"{save_dir}/best.weights"
# model.save_weights(ckpt_path)          # 또는 model.save_weights(ckpt_path, save_format='tf')


In [ ]:
# # # 로드: 동일한 모델 구조를 코드로 재생성한 뒤
# model2 = DeepFM(linear_feature_columns, dnn_feature_columns, task='binary')
# model2.compile(optimizer='adam', loss='binary_crossentropy')
# model2.load_weights(ckpt_path)

# 제출 파일 만들기

In [52]:
# # (앞에서 만든 is_train_num 기반 te_mask 사용)

test_idx = df.index[te_mask]  # 원래 순서 유지가 안전
ids  = df.loc[test_idx, "ID"].astype(str).to_numpy()

# 제출 파일 생성 (ID, clicked 두 컬럼만)
sub = pd.DataFrame({"ID": ids, "clicked": p_test_raw_ens}) # or p_test_cal
sub.to_csv("submission.csv", index=False)  # 순서 유지, 헤더 포함

print(sub.shape, sub.dtypes)
print(sub.head())


(1527298, 2) ID          object
clicked    float32
dtype: object
             ID   clicked
0  TEST_0000000  0.478372
1  TEST_0000001  0.544029
2  TEST_0000002  0.469089
3  TEST_0000003  0.562559
4  TEST_0000004  0.482176
